# Demo del agente: video de la defensa

Este notebook es solo para grabar el video. Hace trece preguntas al agente y muestra,
en cada una, qué herramienta eligió. La mayoría va en una conversación nueva; la 3 sigue la conversación de la 2 y la 12 la retoma después de reiniciar el agente. El trabajo completo está en `obligatorio_agentic_rag.ipynb`.

| Sección del notebook principal | Qué muestra |
|---|---|
| 1. Preparación | Configuración, clave de Groq y bases disponibles |
| 2. Dataset | Los 6 CSV unidos y sin duplicados: 82.486 correos |
| 3. Chunking y embeddings | Fragmentos de 800 caracteres y similitud con e5-small |
| 4. Base vectorial | ChromaDB: correos y papers, búsqueda semántica y umbral |
| 5. Herramientas | Las seis tools que puede llamar el agente |
| 6. Agente con LangGraph | El grafo ReAct y el prompt de sistema |
| 7. Con y sin contexto | La misma pregunta con RAG y sin RAG |
| 8. Pruebas del chatbot | 12 casos y si eligió la tool correcta |
| 9. Memoria y sesiones | Corto y largo plazo, retomar y usuarios en paralelo |
| 10. Chatbot | El chat por consola con sus comandos |
| 11. Limitaciones | Lo que el agente todavía no resuelve |
| 12. Experimentos | Chunking, recuperación, embeddings, API y comparación de LLMs |

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Salida limpia para el video: sin barras de progreso ni avisos de Hugging Face.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import transformers
from IPython.display import Markdown, display

transformers.logging.disable_progress_bar()
sys.path.insert(0, str(Path.cwd()))

from src import agente as modulo_agente
from src import chatbot, config, memoria, recuperacion

# Memoria temporal: la demo arranca sin recuerdos y no toca memoria.sqlite3.
config.ARCHIVO_MEMORIA = Path(tempfile.mkdtemp()) / "memoria_demo.sqlite3"
agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
USUARIO = "nico"

# Carga ahora el modelo de embeddings, los índices y el dataset, para que no aparezca en las respuestas.
recuperacion._indice(config.COLECCION_PAPERS)
recuperacion._indice(config.COLECCION_DATASET)
recuperacion._tabla_dataset()


def preguntar(texto, conversacion, usuario=USUARIO):
    """Envía un mensaje al agente y muestra la respuesta junto con las tools que usó."""
    # Cada conversación es un thread_id distinto, como en el chat con /nueva.
    sesion = {"configurable": {"thread_id": f"{usuario}-{conversacion}", "user_id": usuario}}
    mensajes = agente.invoke({"messages": [("user", texto)]}, sesion)["messages"]
    tools = chatbot._herramientas_usadas(mensajes)
    display(Markdown(f"**{usuario} (conversación {conversacion}):** {texto}\n\n"
                     f"**Tools usadas:** `{', '.join(tools) or 'ninguna'}`\n\n"
                     f"**Agente:** {mensajes[-1].content}"))


print("Agente listo")

## 1. Small-talk
Un saludo no necesita buscar nada: responde directo, sin herramientas.

In [ ]:
preguntar("¡Hola! ¿Qué podés hacer?", conversacion=1)

## 2. Pregunta sobre un paper
El agente busca en los papers indexados en ChromaDB.

In [ ]:
preguntar("¿Cuántos parámetros tiene GPT-3?", conversacion=2)

## 3. Seguimiento en la misma conversación
Memoria de corto plazo: «lo» se entiende como GPT-3 por el historial de la conversación 2.

In [ ]:
preguntar("¿Y con cuántos tokens lo entrenaron?", conversacion=2)

## 4. Pregunta en inglés
Responde en el idioma de la pregunta.

In [ ]:
preguntar("What does the Scaling Laws paper say about how the loss changes with model size?", conversacion=3)

## 5. Resumen de un documento largo
Para resumir trae más fragmentos del mismo paper.

In [ ]:
preguntar("Resumí el paper Attention Is All You Need en tres párrafos.", conversacion=4)

## 6. Estadística exacta del dataset
Para contar no sirve la búsqueda semántica: usa pandas sobre los 82.486 correos.

In [ ]:
preguntar("¿Cuántos correos de phishing hay en el dataset?", conversacion=5)

## 7. Ranking del dataset
También con pandas, sobre el dataset completo.

In [ ]:
preguntar("¿Cuáles son los 5 dominios que más correos de phishing mandan?", conversacion=6)

## 8. Comparar un correo sospechoso
Busca correos parecidos en el dataset y compara.

In [ ]:
preguntar("Me llegó este correo: 'Dear customer, your bank account has been suspended. Click here to verify your password within 24 hours.' ¿Se parece a los phishing del dataset?", conversacion=7)

## 9. Pregunta sin respuesta en los datos
Ningún fragmento supera el umbral de relevancia, así que no inventa.

In [ ]:
preguntar("¿Qué dice el paper de Attention sobre el mundial de fútbol 2022?", conversacion=8)

## 10. Guardar un recuerdo
El agente decide guardar el dato en la memoria de largo plazo, asociada al usuario.

In [ ]:
preguntar("Hola! Soy Nico y trabajo en ciberseguridad.", conversacion=9)
print("Recuerdos guardados:", memoria.leer_recuerdos(agente.store, USUARIO))

## 11. Conversación nueva, mismo usuario
Otro `thread_id`, sin historial. Lo que sabe de Nico viene de la memoria de largo plazo.

In [ ]:
preguntar("¿Te acordás cómo me llamo y a qué me dedico?", conversacion=10)

## 12. Reiniciar el agente y retomar una conversación
Se vuelve a construir el agente desde cero, como si se cerrara el programa. La conversación 2 sigue guardada en SQLite y se retoma sin perder el contexto.

In [ ]:
agente = modulo_agente.construir_agente(memoria.crear_memoria_corto_plazo(), memoria.crear_memoria_largo_plazo())
print("Agente reiniciado")
preguntar("¿Qué te había preguntado antes?", conversacion=2)

## 13. Otro usuario
Cada usuario tiene sus propios recuerdos: Ana no sabe nada de lo que contó Nico.

In [ ]:
preguntar("¿Sabés cómo me llamo?", conversacion=1, usuario="ana")
print("Recuerdos de ana:", memoria.leer_recuerdos(agente.store, "ana"))